# Portfolio Explorer
Análisis visual completo: precios, retornos, riesgo, optimización y efficient frontier.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.gridspec import GridSpec

from src.utils.config import load_config, get_active_portfolios
from src.data.fetcher import DataFetcher
from src.analysis.portfolio import (
    portfolio_returns, summary_stats, correlation_matrix,
    drawdown_series, annualized_return, annualized_volatility, sharpe_ratio
)
from src.analysis.risk import compute_risk_table
from src.analysis.optimization import run_optimization, optimization_summary, efficient_frontier

sns.set_theme(style='darkgrid', palette='tab10')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['figure.figsize'] = (14, 5)

COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
cfg = load_config('../config/portfolio.yaml')
portfolios = get_active_portfolios(cfg)
port_name = list(portfolios.keys())[0]
pcfg = portfolios[port_name]

fetcher = DataFetcher(
    cache_dir='../data/cache',
    expiry_days=cfg['data']['cache_expiry_days']
)
prices = fetcher.get_prices(pcfg['tickers'], pcfg['start_date'], pcfg['end_date'])
returns = fetcher.get_returns(prices)

TICKERS = list(returns.columns)
RFR = cfg['optimization']['risk_free_rate']
N = len(TICKERS)
EQ_WEIGHTS = np.ones(N) / N

print(f"Portfolio: {port_name}")
print(f"Tickers  : {TICKERS}")
print(f"Period   : {returns.index[0].date()} → {returns.index[-1].date()} ({len(returns)} trading days)")

## 1. Precio normalizado (base 100)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
norm = prices / prices.iloc[0] * 100
for i, col in enumerate(norm.columns):
    ax.plot(norm.index, norm[col], label=col, linewidth=1.5)
ax.axhline(100, color='white', linewidth=0.5, linestyle='--', alpha=0.4)
ax.set_title('Precio Normalizado (base 100)', fontsize=14, fontweight='bold')
ax.set_ylabel('Índice (100 = inicio)')
ax.legend(ncol=5, fontsize=9)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f'))
plt.tight_layout()
plt.show()

## 2. Resumen estadístico — retorno vs volatilidad (riesgo-retorno scatter)

In [ ]:
stats = summary_stats(returns, EQ_WEIGHTS, risk_free_rate=RFR)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Scatter riesgo-retorno ---
ax = axes[0]
asset_stats = stats.drop('Portfolio', errors='ignore')
for i, ticker in enumerate(asset_stats.index):
    vol = asset_stats.loc[ticker, 'Ann. Volatility']
    ret = asset_stats.loc[ticker, 'Ann. Return']
    ax.scatter(vol, ret, s=120, color=COLORS[i % len(COLORS)], zorder=3)
    ax.annotate(ticker, (vol, ret), textcoords='offset points', xytext=(6, 4), fontsize=9)

# Portfolio equal weight
if 'Portfolio' in stats.index:
    pv = stats.loc['Portfolio', 'Ann. Volatility']
    pr = stats.loc['Portfolio', 'Ann. Return']
    ax.scatter(pv, pr, s=200, marker='*', color='gold', zorder=4, label='Equal Weight')
    ax.annotate('EW', (pv, pr), textcoords='offset points', xytext=(6, 4), fontsize=9, color='gold')

ax.set_xlabel('Volatilidad Anualizada')
ax.set_ylabel('Retorno Anualizado')
ax.set_title('Riesgo vs Retorno', fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()

# --- Sharpe bar chart ---
ax2 = axes[1]
sharpes = asset_stats['Sharpe Ratio'].sort_values(ascending=True)
colors = ['#e74c3c' if s < 0 else '#2ecc71' if s > 1 else '#3498db' for s in sharpes]
bars = ax2.barh(sharpes.index, sharpes.values, color=colors, edgecolor='none')
ax2.axvline(0, color='white', linewidth=0.8)
ax2.set_title('Sharpe Ratio por Activo', fontweight='bold')
ax2.set_xlabel('Sharpe Ratio')
for bar, val in zip(bars, sharpes.values):
    ax2.text(val + 0.03, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()
print(stats.round(3).to_string())

## 3. Matriz de correlación

In [ ]:
corr = correlation_matrix(returns)
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlación de Retornos Diarios', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Distribución de retornos diarios

In [ ]:
from scipy import stats as scipy_stats

ncols = 5
nrows = int(np.ceil(N / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    ax = axes[i]
    r = returns[ticker].dropna()
    ax.hist(r, bins=60, density=True, alpha=0.7, color=COLORS[i % len(COLORS)], edgecolor='none')
    # Overlay normal fit
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 200)
    mu_fit, std_fit = scipy_stats.norm.fit(r)
    ax.plot(x, scipy_stats.norm.pdf(x, mu_fit, std_fit), 'w--', linewidth=1.5, label='Normal')
    ax.set_title(ticker, fontweight='bold')
    ax.set_xlabel('Retorno diario')
    skew = r.skew()
    kurt = r.kurt()
    ax.text(0.03, 0.95, f'sk={skew:.2f}\nkt={kurt:.2f}', transform=ax.transAxes,
            fontsize=7.5, va='top', color='white')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribución de Retornos Diarios', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Drawdown histórico

In [ ]:
port_ret_series = portfolio_returns(returns, EQ_WEIGHTS)
dd_port = drawdown_series(port_ret_series)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- Drawdown del portafolio EW ---
axes[0].fill_between(dd_port.index, dd_port.values * 100, 0, alpha=0.6, color='#e74c3c')
axes[0].plot(dd_port.index, dd_port.values * 100, color='#e74c3c', linewidth=1)
axes[0].set_title('Drawdown — Portafolio Equal Weight', fontweight='bold')
axes[0].set_ylabel('Drawdown (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

# --- Drawdown por activo ---
for i, ticker in enumerate(TICKERS):
    dd = drawdown_series(returns[ticker])
    axes[1].plot(dd.index, dd.values * 100, label=ticker, linewidth=1, alpha=0.8)
axes[1].set_title('Drawdown por Activo', fontweight='bold')
axes[1].set_ylabel('Drawdown (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend(ncol=5, fontsize=9)

plt.tight_layout()
plt.show()

## 6. VaR / CVaR — comparativa visual

In [ ]:
risk_cfg = cfg['risk']
risk_table = compute_risk_table(
    returns, EQ_WEIGHTS,
    confidence_levels=risk_cfg['var_confidence'],
    horizons=risk_cfg['horizon_days'],
    methods=risk_cfg['var_methods'],
    n_simulations=risk_cfg['mc_simulations'],
)

# Plot VaR a 1 día para los 3 métodos y 2 confidence levels
methods = ['historical', 'parametric', 'monte_carlo']
confs = risk_cfg['var_confidence']
horizons_plot = risk_cfg['horizon_days']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(horizons_plot))
width = 0.25

for ci, conf in enumerate(confs):
    ax = axes[ci]
    for mi, method in enumerate(methods):
        vals = [risk_table.loc[(method, 'VaR'), (conf, h)] * 100 for h in horizons_plot]
        bars = ax.bar(x + mi * width, vals, width, label=method, alpha=0.85)
    ax.set_xticks(x + width)
    ax.set_xticklabels([f'{h}d' for h in horizons_plot])
    ax.set_title(f'VaR al {int(conf*100)}% — por horizonte', fontweight='bold')
    ax.set_ylabel('VaR (%)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend()

plt.tight_layout()
plt.show()

print('\nTabla completa VaR/CVaR:')
display(risk_table.round(4) * 100)

## 7. Optimización — pesos óptimos y Efficient Frontier

In [ ]:
opt_cfg = cfg['optimization']
opt_results = run_optimization(
    returns,
    objectives=opt_cfg['objectives'],
    weight_bounds=tuple(opt_cfg['weight_bounds']),
    risk_free_rate=RFR,
)
opt_sum = optimization_summary(opt_results)

# Efficient frontier
print('Calculando efficient frontier...')
ef = efficient_frontier(
    returns,
    n_points=80,
    weight_bounds=tuple(opt_cfg['weight_bounds']),
    risk_free_rate=RFR,
)

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(1, 2, figure=fig, width_ratios=[1.4, 1])

# --- Efficient Frontier ---
ax1 = fig.add_subplot(gs[0])
sc = ax1.scatter(
    ef['volatility'] * 100, ef['expected_return'] * 100,
    c=ef['sharpe_ratio'], cmap='RdYlGn', s=20, zorder=2
)
plt.colorbar(sc, ax=ax1, label='Sharpe Ratio')

# Activos individuales
asset_stats = stats.drop('Portfolio', errors='ignore')
for i, ticker in enumerate(asset_stats.index):
    vol = asset_stats.loc[ticker, 'Ann. Volatility'] * 100
    ret = asset_stats.loc[ticker, 'Ann. Return'] * 100
    ax1.scatter(vol, ret, s=80, color=COLORS[i % len(COLORS)], zorder=3, alpha=0.9)
    ax1.annotate(ticker, (vol, ret), textcoords='offset points', xytext=(5, 3), fontsize=8)

# Portafolios óptimos
markers = {'max_sharpe': ('*', 'gold', 220), 'min_variance': ('D', 'cyan', 80), 'risk_parity': ('s', 'orange', 80)}
for obj, (mk, color, sz) in markers.items():
    if obj in opt_results:
        r = opt_results[obj]
        ax1.scatter(r['volatility']*100, r['expected_return']*100, marker=mk, color=color, s=sz, zorder=5, label=obj)

# Línea de Capital Market (CML desde risk-free)
if 'max_sharpe' in opt_results:
    ms = opt_results['max_sharpe']
    x_cml = np.linspace(0, ms['volatility'] * 1.2 * 100, 100)
    y_cml = RFR * 100 + ms['sharpe_ratio'] * x_cml
    ax1.plot(x_cml, y_cml, 'w--', linewidth=1, alpha=0.5, label='CML')

ax1.set_xlabel('Volatilidad Anualizada (%)')
ax1.set_ylabel('Retorno Esperado (%)')
ax1.set_title('Efficient Frontier', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)

# --- Pesos óptimos (stacked bar) ---
ax2 = fig.add_subplot(gs[1])
weight_data = {}
for obj in opt_cfg['objectives']:
    if obj in opt_results:
        weight_data[obj] = opt_results[obj]['weights']

weight_df = pd.DataFrame(weight_data, index=TICKERS).T * 100
bottom = np.zeros(len(weight_df))
for i, ticker in enumerate(TICKERS):
    vals = weight_df[ticker].values
    bars = ax2.bar(weight_df.index, vals, bottom=bottom, color=COLORS[i % len(COLORS)],
                   label=ticker, edgecolor='none', alpha=0.9)
    for j, (bar, val) in enumerate(zip(bars, vals)):
        if val > 3:
            ax2.text(bar.get_x() + bar.get_width()/2, bottom[j] + val/2,
                     f'{val:.0f}%', ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')
    bottom += vals

ax2.set_ylabel('Peso (%)')
ax2.set_title('Pesos Óptimos por Estrategia', fontweight='bold')
ax2.legend(loc='upper right', fontsize=8, ncol=2)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

print('\nResumen de portafolios óptimos:')
display(opt_sum[['Ann. Return','Ann. Volatility','Sharpe Ratio']].round(3).style.format('{:.1%}', subset=['Ann. Return','Ann. Volatility']).format('{:.3f}', subset=['Sharpe Ratio']))

## 8. Rolling Sharpe & Volatilidad (ventana 63 días = ~1 trimestre)

In [ ]:
WINDOW = 63

port_ret_series = portfolio_returns(returns, EQ_WEIGHTS)

rolling_vol = port_ret_series.rolling(WINDOW).std() * np.sqrt(252) * 100
rolling_ret = port_ret_series.rolling(WINDOW).mean() * 252
rolling_sharpe = (rolling_ret - RFR) / (rolling_vol / 100)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(rolling_vol.index, rolling_vol, color='#e67e22', linewidth=1.5)
axes[0].fill_between(rolling_vol.index, rolling_vol, alpha=0.2, color='#e67e22')
axes[0].set_title(f'Volatilidad Rolling ({WINDOW}d) — Portafolio EW', fontweight='bold')
axes[0].set_ylabel('Vol Anualizada (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

axes[1].plot(rolling_sharpe.index, rolling_sharpe, color='#3498db', linewidth=1.5)
axes[1].axhline(0, color='white', linewidth=0.6, linestyle='--')
axes[1].axhline(1, color='#2ecc71', linewidth=0.6, linestyle='--', alpha=0.6)
axes[1].fill_between(rolling_sharpe.index, rolling_sharpe, 0,
                     where=rolling_sharpe >= 0, alpha=0.2, color='#2ecc71')
axes[1].fill_between(rolling_sharpe.index, rolling_sharpe, 0,
                     where=rolling_sharpe < 0, alpha=0.2, color='#e74c3c')
axes[1].set_title(f'Sharpe Ratio Rolling ({WINDOW}d)', fontweight='bold')
axes[1].set_ylabel('Sharpe Ratio')

plt.tight_layout()
plt.show()